# Assignment 2: End-to-End Regression Project
## Insurance Cost Prediction — From Raw Data to a Deployable Model

**Student:** Arslan Hashmi  
**Internship ID:** ZYNVEX-CERT-0486  
**Institution:** National University of Technology (NUTECH)  
**Track:** AI/ML Internship | Week 2 Weekend Assignment

---

This notebook implements a complete regression pipeline:
1. Data Loading & Exploration
2. Feature Engineering
3. Model Building (Simple Linear, Multiple Linear, Polynomial)
4. Model Evaluation & Comparison
5. Model Selection & Interpretation
6. Lightweight Deployment

## 1. Data Loading & Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('Libraries loaded successfully')

In [ ]:
# Upload insurance.csv in Colab first, then run:
df = pd.read_csv('insurance.csv')
print('Shape:', df.shape)
print('\nColumn types:')
print(df.dtypes)
print('\nFirst 5 rows:')
df.head()

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nSummary statistics:')
df.describe()

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
print('Shape after removing duplicate:', df.shape)

### Target Variable Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['charges'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Distribution of Insurance Charges (Original)')
axes[0].set_xlabel('Charges ($)')
sns.histplot(np.log1p(df['charges']), kde=True, ax=axes[1], color='seagreen')
axes[1].set_title('Distribution of log1p(Charges)')
axes[1].set_xlabel('log1p(Charges)')
plt.tight_layout()
plt.show()
print(f"Skewness of charges      : {df['charges'].skew():.3f}  (right-skewed)")
print(f"Skewness of log1p(charges): {np.log1p(df['charges']).skew():.3f}  (nearly symmetric)")

**Observation:** The target `charges` is strongly right-skewed (skew ~1.52). A log transformation (`log1p`) makes the distribution nearly symmetric.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sns.boxplot(x='smoker', y='charges', data=df, ax=axes[0], palette='Set2')
axes[0].set_title('Charges by Smoker Status')
sns.scatterplot(x='age', y='charges', hue='smoker', data=df, ax=axes[1], alpha=0.6)
axes[1].set_title('Charges vs Age (colored by Smoker)')
sns.scatterplot(x='bmi', y='charges', hue='smoker', data=df, ax=axes[2], alpha=0.6)
axes[2].set_title('Charges vs BMI (colored by Smoker)')
plt.tight_layout()
plt.show()

**Key Insights:**
- Smoker status is the strongest driver of insurance cost.
- Age shows a clear positive trend with charges.
- BMI has a visible relationship, especially among smokers.
- Smoking + high BMI / older age produces the highest charges.

## 2. Feature Engineering

In [ ]:
df['bmi_category'] = pd.cut(df['bmi'], bins=[0, 18.5, 25, 30, 100],
                            labels=['underweight', 'normal', 'overweight', 'obese'])
df['smoker_flag'] = (df['smoker'] == 'yes').astype(int)
df['smoker_age'] = df['smoker_flag'] * df['age']
df['log_charges'] = np.log1p(df['charges'])
print('New features created: bmi_category, smoker_flag, smoker_age, log_charges')
df[['bmi', 'bmi_category', 'smoker', 'age', 'smoker_age']].head(8)

In [ ]:
df_model = df.copy()
df_model['sex'] = (df_model['sex'] == 'male').astype(int)
df_model = pd.get_dummies(df_model, columns=['region', 'bmi_category'], drop_first=True)
feature_cols = [c for c in df_model.columns if c not in ['charges', 'log_charges', 'smoker']]
print('Final feature set ({} features):'.format(len(feature_cols)))
print(feature_cols)
X = df_model[feature_cols]
y = df_model['charges']
y_log = df_model['log_charges']

**Encoding justification:**
- `sex` and `smoker` are binary → 0/1 mapping.
- `region` and `bmi_category` have multiple unordered levels → one-hot encoding.
- `smoker_age` interaction captures the non-linear cost jump for older smokers.
- Log target is used for one model because of strong right skew.

## 3. Model Building

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
_, _, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.2, random_state=42)
print(f'Training samples: {X_train.shape[0]}')
print(f'Test samples:     {X_test.shape[0]}')

### 3.1 Simple Linear Regression (most predictive feature = smoker)

In [ ]:
simple_model = LinearRegression()
simple_model.fit(X_train[['smoker_flag']], y_train)
y_pred_simple = simple_model.predict(X_test[['smoker_flag']])
print(f'Simple Linear (smoker only) — R²: {r2_score(y_test, y_pred_simple):.4f}')

### 3.2 Multiple Linear Regression (all engineered features)

In [ ]:
multi_model = LinearRegression()
multi_model.fit(X_train, y_train)
y_pred_multi = multi_model.predict(X_test)
print(f'Multiple Linear — R²: {r2_score(y_test, y_pred_multi):.4f}')

### 3.3 Multiple Linear on log-transformed target

In [ ]:
multi_log_model = LinearRegression()
multi_log_model.fit(X_train, y_train_log)
y_pred_multi_log = np.expm1(multi_log_model.predict(X_test))
print(f'Multiple Linear (log target) — R²: {r2_score(y_test, y_pred_multi_log):.4f}')

### 3.4 Polynomial Regression (degree 2 on key features)

In [ ]:
key_features = ['age', 'bmi', 'children', 'smoker_flag', 'smoker_age']
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train[key_features])
X_test_poly  = poly.transform(X_test[key_features])
poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train)
y_pred_poly = poly_model.predict(X_test_poly)
print(f'Polynomial (degree=2) — R²: {r2_score(y_test, y_pred_poly):.4f}')

## 4. Model Evaluation

In [ ]:
def evaluate(y_true, y_pred, name):
    return {
        'Model': name,
        'R2': round(r2_score(y_true, y_pred), 4),
        'MAE': round(mean_absolute_error(y_true, y_pred), 2),
        'MSE': round(mean_squared_error(y_true, y_pred), 2),
        'RMSE': round(np.sqrt(mean_squared_error(y_true, y_pred)), 2)
    }

results = [
    evaluate(y_test, y_pred_simple, 'Simple Linear (smoker only)'),
    evaluate(y_test, y_pred_multi, 'Multiple Linear'),
    evaluate(y_test, y_pred_multi_log, 'Multiple Linear (log target)'),
    evaluate(y_test, y_pred_poly, 'Polynomial Regression (deg=2)')
]
comparison = pd.DataFrame(results).sort_values('R2', ascending=False).reset_index(drop=True)
print('Model Comparison Table')
print('=' * 70)
print(comparison.to_string(index=False))

In [ ]:
poly_pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('lr', LinearRegression())
])
cv_scores = cross_val_score(poly_pipe, X[key_features], y, cv=5, scoring='r2')
print('5-Fold Cross-Validation (Polynomial model)')
print(f'Individual fold R2 : {np.round(cv_scores, 4)}')
print(f'Mean R2           : {cv_scores.mean():.4f}')
print(f'Std R2            : {cv_scores.std():.4f}')

## 5. Model Selection & Interpretation

**Best Model: Polynomial Regression (degree=2)** on key features `[age, bmi, children, smoker_flag, smoker_age]`.

**Justification:**
- Highest test R2 (~0.88) and lowest MAE / RMSE.
- Polynomial terms capture non-linear effects (especially smoking x age/BMI) that pure linear models miss.
- Cross-validation mean R2 remains high and stable.

**Feature Impact (from Multiple Linear coefficients):**

In [ ]:
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': multi_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)
print('Top features by absolute coefficient magnitude:')
print(coef_df.head(10).to_string(index=False))

**Interpretation:**
- `smoker_flag` has the largest positive coefficient — matches the strong visual separation.
- `smoker_age` interaction is highly influential (cost penalty of smoking grows with age).
- `age` and `bmi` contribute positively as expected.
- These findings align with the exploratory analysis.

## 6. Lightweight Model Deployment

In [ ]:
deployment_artifacts = {
    'poly': poly,
    'model': poly_model,
    'key_features': key_features
}
joblib.dump(deployment_artifacts, 'insurance_cost_model.joblib')
print('Model saved to insurance_cost_model.joblib')

In [ ]:
def predict_insurance_cost(age, sex, bmi, children, smoker, region):
    """
    Predict medical insurance cost for a new person.
    """
    artifacts = joblib.load('insurance_cost_model.joblib')
    smoker_flag = 1 if str(smoker).lower() == 'yes' else 0
    smoker_age  = smoker_flag * age
    row = pd.DataFrame([{
        'age': age,
        'bmi': bmi,
        'children': children,
        'smoker_flag': smoker_flag,
        'smoker_age': smoker_age
    }])
    X_poly = artifacts['poly'].transform(row)
    prediction = artifacts['model'].predict(X_poly)[0]
    return max(0.0, float(prediction))


print('Example 1: 25-year-old non-smoker, normal BMI')
print(f"Predicted cost: ${predict_insurance_cost(25, 'female', 22.5, 0, 'no', 'northeast'):,.2f}")

print('\nExample 2: 55-year-old smoker, obese')
print(f"Predicted cost: ${predict_insurance_cost(55, 'male', 36.0, 2, 'yes', 'southeast'):,.2f}")

---
## Summary

| Step | What was done |
|------|---------------|
| Exploration | Strong effect of smoking, age, BMI; charges are right-skewed |
| Feature Engineering | Binary encoding, one-hot, smoker x age interaction, log target |
| Models | Simple Linear, Multiple Linear, Multiple Linear (log), Polynomial (deg=2) |
| Best Model | Polynomial Regression (R2 ~ 0.88, lowest MAE/RMSE) |
| Deployment | Model + transformer saved with joblib; reusable prediction function |

**Files produced:**
- `Assignment2_ArslanHashmi.ipynb`
- `insurance_cost_model.joblib`

**Completed by:** Arslan Hashmi  
**Date:** July 2026